In [ ]:
import os
import json
import pandas as pd

json_dir = "/nfs/hongshu/traces/analysis_json"
rows = []

for fname in os.listdir(json_dir):
    if fname.endswith(".json"):
        cluster = fname.split(".")[0]  # Extract 'clusterX' from filename
        download_path = None
        slab_size = 4
        with open(os.path.join(json_dir, fname), "r") as f:
            try:
                data = json.load(f)
                if cluster.startswith("cluster"):
                    cluster = f"twitter_{cluster}"
                    download_path = f"twitter/{fname[:-14]}"
                    slab_size = 1
                elif cluster.startswith("w"):
                    cluster = f"cp_{cluster}"
                    download_path = f"cloudphysics/{fname[:-14]}"
                else:
                    cluster = f"meta_{'_'.join(cluster.split('_')[:2])}"
                    download_path = f"metaKV/{fname[:-14]}"
                data["trace_name"] = cluster
                data["download_path"] = download_path
                data["file_name"] = fname[:-14]
                data["slab_size"] = slab_size
                rows.append(data)

            except json.JSONDecodeError as e:
                print(f"Error decoding JSON from {fname}: {e}")

df = pd.DataFrame(rows)

In [22]:
pd.set_option('display.float_format', '{:.4f}'.format)
df['wss'] = df['number_of_obj_GiB'] + (df['number_of_objects'] * (20 + 32) / (1024 * 1024 * 1024))  # Convert to GiB

In [29]:
[v for v in df['trace_name'].values.tolist() if v.startswith("meta")]

['meta_202312_kv',
 'meta_meta_kvcache',
 'meta_202210_kv',
 'meta_202206_kv',
 'meta_202401_kv']

In [26]:
df.to_csv("trace_info.csv", index=False)

In [17]:
"""
Slab classes start at 64 bytes and exponentially increase in size by a factor of 1.07 up to 1 MB, aligned on
4-byte boundaries3
"""
import numpy as np

df['min_item_size'] = np.maximum(df['min_req_size'], 24) + 20 + 32
df['max_item_size'] = df['max_req_size'] + 20 + 32


In [18]:
print(f"min item size, min{df['min_item_size'].min()}, max {df['min_item_size'].max()}")
print(f"max item size, min{df['max_item_size'].min()}, max {df['max_item_size'].max()}")

min item size, min76, max 120
max item size, min94, max 1010406


In [6]:
import numpy as np


def find_slab_classes(min_size=64, max_size=1024 * 1024, step_factor=1.07):
    sizes = []
    size = min_size

    while size <= max_size:
        # Align to 4 bytes
        aligned_size = int(np.ceil(size / 4.0)) * 4
        if len(sizes) == 0 or aligned_size != sizes[-1]:
            sizes.append(aligned_size)
        size *= step_factor
    
    return sizes

In [21]:
classes = find_slab_classes(72, 1024 * 1024, 1.25)
print("Slab classes:", classes)
print(len(classes), "classes")

Slab classes: [72, 92, 116, 144, 176, 220, 276, 344, 432, 540, 672, 840, 1048, 1312, 1640, 2048, 2560, 3200, 4000, 5000, 6248, 7808, 9760, 12200, 15248, 19060, 23824, 29780, 37224, 46532, 58164, 72704, 90880, 113596, 141996, 177496, 221868, 277336, 346668, 433336, 541668, 677088, 846356]
43 classes


In [14]:
max_sizes = [val + 32 + 20 for val in df['max_req_size'].values.tolist()]
print("above 512kb: ", sum([1 for val in max_sizes if val > 512 * 1024]))
print("above 256kb: ", sum([1 for val in max_sizes if val > 256 * 1024]))
print("above 128kb: ", sum([1 for val in max_sizes if val > 128 * 1024]))
print("above 64kb: ", sum([1 for val in max_sizes if val > 64 * 1024]))
print("above 32kb: ", sum([1 for val in max_sizes if val > 32 * 1024]))
print("above 16kb: ", sum([1 for val in max_sizes if val > 16 * 1024]))
print("above 8kb: ", sum([1 for val in max_sizes if val > 8 * 1024]))

above 512kb:  3
above 256kb:  5
above 128kb:  11
above 64kb:  13
above 32kb:  18
above 16kb:  23
above 8kb:  29


In [13]:
1024 * 1024 * 0.5

524288.0